In [1]:
%pip install pypdf python-docx beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from isaacus import Isaacus
from dotenv import load_dotenv
import os
import requests
from examples import EXAMPLE_DOCUMENTS
from custom_taxonomy import CUSTOM_TAXONOMY

In [3]:
load_dotenv()  # Load environment variables from `.env` files if any are present.
isaacus_client = Isaacus(api_key=os.getenv("ISAACUS_API_KEY"))

TAXONOMY

TITLE | CLIENT | DATE 

In [4]:
def clean_text(text):
    return text.replace("\n", " ").strip() if text else None

def extract_date(doc):
    for date in doc.dates:
        if date.type == "creation":
            return date.value
    return None

def extract_title(doc):
    return clean_text(doc.title.decode(doc.text)) if doc.title else None

def extract_parties(doc):
    parties = []
    for entity in doc.persons:
        if entity.parent is None:
            entity_name = clean_text(entity.name.decode(doc.text))
            parties.append(entity_name)

    return parties
def extract_locations(doc):
    locations = []
    for entity in doc.locations:
        entity_name = clean_text(entity.name.decode(doc.text))
        locations.append(entity_name)

    return locations
def extract_terms(doc):
    terms = []
    for term in doc.terms:
        term_name = clean_text(term.name.decode(doc.text))
        term_definition = clean_text(term.meaning.decode(doc.text))
        terms.append({"name": term_name, "definition": term_definition})

    return terms

def extradct_signatures(doc):
    signatures = []
    for segment in doc.segments:
        if segment.type == "signature":
            signatures.append(segment.span.decode(doc.text))
    return signatures

def extract_enriched_features_from_document(document_text):
    response = isaacus_client.enrichments.create(
        model="kanon-2-enricher",
        texts=[document_text],
        overflow_strategy="auto",
    )
    doc = response.results[0].document
    features = {}
    features["title"] = extract_title(doc)
    features["parties"] = extract_parties(doc)
    features["date"] = extract_date(doc)
    features["locations"] = extract_locations(doc)
    features["terms"] = extract_terms(doc)
    features["signatures"] = extradct_signatures(doc)
    return features


In [5]:

def score_document_by_category(document_text, query):
    result = isaacus_client.rerankings.create(
        model="kanon-2-reranker",
        query=query,
        texts=[document_text],
    )
    return result.results[0].score

def classify_document(document, nodes, mode="full"):
    if mode == "greedy":
        return classify_greedy(document, nodes)
    if mode == "full":
        return classify_full(document, nodes)

def classify_greedy(document: str, nodes: list[dict]) -> dict:
    scored_nodes = [{
            "node": node,
            "score": score_document_by_category(document, node["description"])
        }
        for node in nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    if not best_node.get("children"):
        return {
            "label": best_node["name"],
            "score": best["score"],
            "description": best_node["description"],
            "mode": "greedy",
        }

    return classify_greedy(document, best_node["children"])


def flatten_taxonomy(nodes):
    all_nodes = []

    for node in nodes:
        all_nodes.append(node)
        children = node.get("children", [])
        if children:
            all_nodes.extend(flatten_taxonomy(children))

    return all_nodes


def classify_full(document, nodes):
    all_nodes = flatten_taxonomy(nodes)

    scored_nodes = [
        {
            "node": node,
            "score": score_document_by_category(document, node["description"]),
        }
        for node in all_nodes
    ]

    best = max(scored_nodes, key=lambda item: item["score"])
    best_node = best["node"]

    return {
        "label": best_node["name"],
        "score": best["score"],
        "description": best_node["description"],
        "mode": "full",
    }



In [6]:
def pretty_title(label, parties, date):
    year = date.split("-")[0] if date else None
    if parties and date:
        pretty_title = f"{label} - {parties[0]} ({year})"
    elif parties:
        pretty_title = f"{label} - {parties[0]}"
    elif date:
        pretty_title= f"{label} ({year[:-2]})"
    return pretty_title


In [7]:
def extract_metadata_from_text(document_text, taxonomy_mode="full"):
    features = extract_enriched_features_from_document(document_text)
    category = classify_document(document_text, CUSTOM_TAXONOMY, mode=taxonomy_mode)
    features["category"] = category
    features["pretty_title"] = pretty_title(category["label"], features["parties"], features["date"])
    print(features)


In [ ]:
import io
import json
import time
import traceback
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display, clear_output
from pypdf import PdfReader
from docx import Document as DocxDocument
from bs4 import BeautifulSoup

# -----------------------------------------------------------------------------
# Required notebook objects
# -----------------------------------------------------------------------------
if "extract_enriched_features_from_document" not in globals():
    raise NameError("extract_enriched_features_from_document is not defined in this notebook.")

if "CUSTOM_TAXONOMY" not in globals():
    raise NameError("CUSTOM_TAXONOMY is not defined in this notebook.")

def classify_document_compat(document_text, taxonomy, mode="full"):
    if "classify_document" in globals():
        return classify_document(document_text, taxonomy, mode=mode)
    if "classify" in globals():
        return classify(document_text, taxonomy, mode=mode)
    raise NameError("Neither classify_document nor classify is defined in this notebook.")

def pretty_title_compat(label, parties, date):
    if "pretty_title" in globals():
        try:
            value = pretty_title(label, parties, date)
            if value:
                return value
        except Exception:
            pass
    if label and parties and isinstance(parties, list) and parties:
        return f"{label} - {parties[0]}"
    return label or "Untitled"

# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def uploaded_items(value):
    if isinstance(value, dict):
        return list(value.values())
    if isinstance(value, (list, tuple)):
        return list(value)
    return []

def to_bytes(content):
    if isinstance(content, bytes):
        return content
    if isinstance(content, memoryview):
        return content.tobytes()
    if isinstance(content, bytearray):
        return bytes(content)
    raise TypeError(f"Unsupported file content type: {type(content)}")

def extract_text_from_pdf(file_bytes):
    reader = PdfReader(io.BytesIO(file_bytes))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n\n".join(pages).strip()

def extract_text_from_docx(file_bytes):
    doc = DocxDocument(io.BytesIO(file_bytes))
    return "\n\n".join(p.text for p in doc.paragraphs if p.text and p.text.strip()).strip()

def extract_text_from_html(file_bytes):
    text = file_bytes.decode("utf-8", errors="ignore")
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text("\n", strip=True)

def extract_text_from_plain(file_bytes):
    try:
        return file_bytes.decode("utf-8")
    except UnicodeDecodeError:
        return file_bytes.decode("latin-1", errors="ignore")

def bytes_to_text(file_bytes, name=""):
    suffix = Path(name).suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_bytes)
    if suffix == ".docx":
        return extract_text_from_docx(file_bytes)
    if suffix in {".html", ".htm"}:
        return extract_text_from_html(file_bytes)
    return extract_text_from_plain(file_bytes)

def log(*args):
    with debug:
        print(*args)

def set_stage(step, total, label):
    stage_progress.max = total
    stage_progress.value = step
    stage_label.value = f"<b>Stage:</b> {label}"

# -----------------------------------------------------------------------------
# Minimal summary table (Python-only, no JS)
# -----------------------------------------------------------------------------
def esc(value):
    import html
    return html.escape("" if value is None else str(value))

def short_list(items, max_items=2):
    items = items or []
    if not items:
        return "—"
    shown = ", ".join(str(x) for x in items[:max_items])
    if len(items) > max_items:
        shown += f" +{len(items) - max_items}"
    return shown

def render_summary_table(records):
    rows = []
    for r in records:
        rows.append(f"""
        <tr>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{esc(r.get('pretty_title') or r.get('title') or r.get('source_name') or 'Untitled')}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{esc((r.get('category') or {}).get('label', '—'))}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{esc(r.get('date') or '—')}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{esc(short_list(r.get('parties', [])))}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{esc(short_list(r.get('locations', []), 1))}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{len(r.get('terms', []) or [])}</td>
            <td style="padding:8px 10px; border-bottom:1px solid #e5e7eb;">{len(r.get('signatures', []) or [])}</td>
        </tr>
        """)

    html = f"""
    <div style="border:1px solid #e5e7eb; border-radius:12px; overflow:auto; background:#fff;">
      <div style="padding:10px 12px; border-bottom:1px solid #e5e7eb; font:600 13px Inter, sans-serif;">
        Summary table · {len(records)} document(s)
      </div>
      <table style="width:100%; border-collapse:collapse; min-width:900px; font:13px Inter, sans-serif;">
        <thead style="background:#f8fafc;">
          <tr>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Title</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Category</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Date</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Parties</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Locations</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Terms</th>
            <th style="padding:8px 10px; text-align:left; border-bottom:1px solid #e5e7eb;">Signatures</th>
          </tr>
        </thead>
        <tbody>
          {''.join(rows) if rows else '<tr><td colspan="7" style="padding:14px;">No records yet.</td></tr>'}
        </tbody>
      </table>
    </div>
    """
    with viewer_out:
        clear_output(wait=True)
        display(HTML(html))

# -----------------------------------------------------------------------------
# Optional full viewer payload test
# -----------------------------------------------------------------------------
def build_payload_stats(records):
    payload = json.dumps(records)
    return {
        "record_count": len(records),
        "json_chars": len(payload),
        "json_kb": round(len(payload.encode("utf-8")) / 1024, 1),
    }

def render_full_viewer_test(_=None):
    t0 = time.time()
    status.value = "<b>Status:</b> Building full viewer payload…"
    set_stage(1, 2, "Building JSON payload")

    stats = build_payload_stats(records)
    log("[render-test] payload stats:", stats)

    # Keep this intentionally minimal: no massive custom JS yet.
    html = f"""
    <div style="border:1px solid #e5e7eb; border-radius:12px; background:#fff; padding:16px; font:13px Inter, sans-serif;">
      <div style="font-weight:600; margin-bottom:8px;">Full viewer test</div>
      <pre style="white-space:pre-wrap; margin:0;">{json.dumps(stats, indent=2)}</pre>
    </div>
    """

    set_stage(2, 2, "Rendering notebook output")
    with viewer_out:
        clear_output(wait=True)
        display(HTML(html))

    status.value = f"<b>Status:</b> Full viewer test rendered in {time.time() - t0:.2f}s"
    stage_label.value = "<b>Stage:</b> Idle"
    stage_progress.bar_style = "success"

# -----------------------------------------------------------------------------
# Ingestion
# -----------------------------------------------------------------------------
records = []

def ingest_file(file_info):
    name = file_info.get("name", "Untitled")
    raw_content = file_info.get("content", b"")
    t0 = time.time()

    set_stage(1, 5, f"Reading bytes for {name}")
    file_bytes = to_bytes(raw_content)

    set_stage(2, 5, f"Extracting text from {name}")
    document_text = bytes_to_text(file_bytes, name=name)
    log(f"[extract] text_length={len(document_text)}")

    if not document_text or len(document_text.strip()) < 20:
        raise ValueError("Very little text was extracted. If this is a scanned PDF, you may need OCR.")

    document_text = document_text[:80000]

    set_stage(3, 5, f"Enriching {name}")
    t_enrich = time.time()
    features = extract_enriched_features_from_document(document_text)
    log(f"[enrich] {time.time() - t_enrich:.2f}s")

    set_stage(4, 5, f"Classifying {name}")
    t_classify = time.time()
    category = classify_document_compat(document_text, CUSTOM_TAXONOMY, mode="full")
    log(f"[classify] {time.time() - t_classify:.2f}s")

    set_stage(5, 5, f"Preparing record for {name}")
    record = {
        "source_name": name,
        "title": features.get("title"),
        "pretty_title": pretty_title_compat(category.get("label"), features.get("parties"), features.get("date")),
        "date": features.get("date"),
        "parties": features.get("parties", []),
        "locations": features.get("locations", []),
        "terms": features.get("terms", []),
        "signatures": features.get("signatures", []),
        "category": category,
        # deliberately no embedded base64 previews in debug mode
        "_text_excerpt": document_text[:3000],
    }

    log(f"[ingest] total {time.time() - t0:.2f}s: {name}")
    return record

# -----------------------------------------------------------------------------
# Widgets
# -----------------------------------------------------------------------------
upload = widgets.FileUpload(
    accept=".txt,.md,.html,.pdf,.docx,.doc",
    multiple=True,
)

file_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=1,
    description="Files",
    bar_style="",
    layout=widgets.Layout(width="260px"),
)

stage_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=5,
    description="Stage",
    bar_style="",
    layout=widgets.Layout(width="260px"),
)

status = widgets.HTML("<b>Status:</b> Ready")
stage_label = widgets.HTML("<b>Stage:</b> Idle")
debug = widgets.Output(layout={"border": "1px solid #e5e7eb", "padding": "8px", "max_height": "180px", "overflow": "auto"})
viewer_out = widgets.Output()
render_button = widgets.Button(description="Render full viewer test", button_style="")

toolbar = widgets.HBox(
    [
        widgets.HTML("<div style='font:600 15px/1.2 Inter,-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;'>Metadata Viewer Debug</div>"),
        widgets.HBox([file_progress, stage_progress, upload, render_button], layout=widgets.Layout(gap="10px", align_items="center")),
    ],
    layout=widgets.Layout(justify_content="space-between", align_items="center", width="100%"),
)

# -----------------------------------------------------------------------------
# Upload handler
# -----------------------------------------------------------------------------
def on_upload(change):
    global records

    new_files = uploaded_items(change["new"])
    total = len(new_files)

    file_progress.max = max(total, 1)
    file_progress.value = 0
    file_progress.bar_style = "info"

    stage_progress.max = 5
    stage_progress.value = 0
    stage_progress.bar_style = "info"

    status.value = f"<b>Status:</b> Ingesting {total} file(s)…"
    stage_label.value = "<b>Stage:</b> Starting"

    try:
        for i, file_info in enumerate(new_files, start=1):
            record = ingest_file(file_info)
            records.append(record)
            file_progress.value = i
            status.value = f"<b>Status:</b> Ingested {i}/{total} file(s)"
        stage_label.value = "<b>Stage:</b> Rendering summary table"
        render_summary_table(records)
        status.value = f"<b>Status:</b> Ready · {len(records)} document(s)"
        file_progress.bar_style = "success"
        stage_progress.bar_style = "success"
    except Exception as e:
        file_progress.bar_style = "danger"
        stage_progress.bar_style = "danger"
        status.value = f"<b>Status:</b> Error: {e}"
        log(traceback.format_exc())

    try:
        upload.value = ()
        upload._counter = 0
    except Exception:
        pass

try:
    upload.unobserve_all()
except Exception:
    pass

upload.observe(on_upload, names="value")
render_button.on_click(render_full_viewer_test)

display(toolbar)
display(status)
display(stage_label)
display(debug)
display(viewer_out)

render_summary_table(records)

HTML(value='<b>Status:</b> Ready')

HTML(value='<b>Stage:</b> Idle')

Output(layout=Layout(border_bottom='1px solid #e5e7eb', border_left='1px solid #e5e7eb', border_right='1px sol…

Output()

In [ ]:
import json
from pathlib import Path
from IPython.display import HTML, display

def flatten_leaf_labels(nodes):
    labels = []
    for node in nodes:
        children = node.get("children", [])
        if children:
            labels.extend(flatten_leaf_labels(children))
        else:
            labels.append(node["name"])
    return labels

category_options = sorted(set(flatten_leaf_labels(CUSTOM_TAXONOMY)))

template = Path("metadata_viewer_interactive.html").read_text(encoding="utf-8")

html = (
    template
    .replace("__DATA_JSON__", json.dumps(records).replace("</", "<\\/"))
    .replace("__CATEGORY_OPTIONS__", json.dumps(category_options).replace("</", "<\\/"))
)

display(HTML(html))

FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\metadata_viewer_interactive.html'